# NFL Roster Construction & Cap Efficiency
### Data Collection: 2013-2025

Pulls player stats, rosters, and contracts, merges them with the same
leakage-safe logic used in the NFL Player Value Engine, then aggregates
to team + position group + season for cap allocation analysis.

In [ ]:
import nfl_data_py as nfl
import pandas as pd

YEARS = list(range(2013, 2026))  # 2013-2025, avoids incomplete current-season files

print("Pulling player stats...")
stats = nfl.import_seasonal_data(years=YEARS)
print(f"Stats shape: {stats.shape}")

In [ ]:
print("Pulling roster data...")
rosters = nfl.import_seasonal_rosters(years=YEARS)
print(f"Rosters shape: {rosters.shape}")

In [ ]:
# Keep only the columns needed from rosters
roster_clean = rosters[['player_id', 'season', 'player_name', 'position',
                          'age', 'years_exp', 'draft_number', 'entry_year',
                          'team', 'weight', 'height']].copy()

roster_clean = roster_clean.drop_duplicates(subset=['player_id', 'season'])

df = pd.merge(stats, roster_clean, on=['player_id', 'season'], how='inner')
print(f"Merged stats+roster shape: {df.shape}")
print(f"Season range: {df['season'].min()} - {df['season'].max()}")

In [ ]:
print("Pulling contract data...")
contracts = nfl.import_contracts()
print(f"Contracts shape: {contracts.shape}")

contracts_clean = contracts[['player', 'position', 'team', 'year_signed',
                               'years', 'value', 'apy', 'guaranteed',
                               'draft_overall']].copy()

contracts_clean = contracts_clean.rename(columns={'player': 'player_name', 'apy': 'aav'})

# Drop rows with missing/placeholder year_signed (e.g. 0) before merging
contracts_clean = contracts_clean[contracts_clean['year_signed'] > 2000]
print(f"Contracts after cleaning: {contracts_clean.shape}")

### Leakage-safe merge

Same approach as the Player Value Engine: only match a contract to a season
if it was signed on or before that season, and keep the most recent
qualifying contract per player per season.

In [ ]:
master_df = pd.merge(df, contracts_clean, on=['player_name', 'position'], how='inner')
master_df = master_df[master_df['year_signed'] <= master_df['season']]
master_df = master_df.sort_values('year_signed', ascending=False)
master_df = master_df.drop_duplicates(subset=['player_name', 'season'], keep='first')

print(f"Master dataset shape: {master_df.shape}")
print(f"Season range: {master_df['season'].min()} - {master_df['season'].max()}")

### Position group mapping

Rolling individual positions up to standard front-office position groups
so cap allocation can be compared at a meaningful level (e.g. all offensive
linemen together, not split by T/G/C).

In [ ]:
position_group_map = {
    'QB': 'QB',
    'RB': 'RB', 'FB': 'RB',
    'WR': 'WR',
    'TE': 'TE',
    'T': 'OL', 'G': 'OL', 'C': 'OL', 'OL': 'OL', 'OT': 'OL', 'OG': 'OL',
    'DE': 'DL', 'DT': 'DL', 'NT': 'DL', 'DL': 'DL',
    'LB': 'LB', 'OLB': 'LB', 'ILB': 'LB', 'MLB': 'LB',
    'CB': 'CB', 'DB': 'CB',
    'S': 'S', 'FS': 'S', 'SS': 'S',
    'K': 'ST', 'P': 'ST', 'LS': 'ST',
}

master_df['position_group'] = master_df['position'].map(position_group_map)

unmapped = master_df[master_df['position_group'].isna()]['position'].unique()
print(f"Unmapped positions (check these): {unmapped}")

In [ ]:
# Team-level cap allocation by position group and season
team_cap_allocation = (
    master_df.groupby(['team_x', 'season', 'position_group'])['aav']
    .sum()
    .reset_index()
    .rename(columns={'team_x': 'team', 'aav': 'total_cap_allocated'})
)

print(f"Team cap allocation shape: {team_cap_allocation.shape}")
team_cap_allocation.head(10)

### Team performance data

Pulling game results to compute wins and point differential per team per
season — this becomes the outcome variable for the efficiency analysis.

In [ ]:
print("Pulling schedule/results data...")
schedules = nfl.import_schedules(years=YEARS)
print(f"Schedules shape: {schedules.shape}")
schedules[['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].head()

In [ ]:
# Build a team-season win/point-differential table from schedule results
completed = schedules.dropna(subset=['home_score', 'away_score']).copy()

home = completed[['season', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'points_for', 'away_score': 'points_against'})
away = completed[['season', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'points_for', 'home_score': 'points_against'})

team_games = pd.concat([home, away], ignore_index=True)
team_games['win'] = (team_games['points_for'] > team_games['points_against']).astype(int)

team_performance = team_games.groupby(['team', 'season']).agg(
    wins=('win', 'sum'),
    games=('win', 'count'),
    point_diff=('points_for', lambda x: x.sum() - team_games.loc[x.index, 'points_against'].sum())
).reset_index()

print(f"Team performance shape: {team_performance.shape}")
team_performance.head(10)

In [ ]:
# Save intermediate outputs to data folder
team_cap_allocation.to_csv('../data/team_cap_allocation.csv', index=False)
team_performance.to_csv('../data/team_performance.csv', index=False)

print("Saved team_cap_allocation.csv and team_performance.csv")